In [7]:
#import shutil
#shutil.rmtree("Feature_Targets")
#!pip install --user duckdb

In [1]:
import pandas as pd
import numpy as np
import os
import time
from tqdm.auto import tqdm
import duckdb
import sys

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_preprocessing as du

importlib.reload(du)

<module 'utils.data_preprocessing' from '/home/david/Documents/Masterarbeit/Nonlinear-Return-Predictability-of-Alternative-Limit-Order-Book-Price-Measures/utils/data_preprocessing.py'>

In [2]:
# Initialize notebook

# get parent dict
parent = os.path.dirname(os.getcwd())

# set up duckdb connection
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

In [ ]:
# load fut-data into memory, as it is used across all symbols
fut_dict = {}

for date in tqdm(du.SAMPLE_DATES[:10], desc="Dates"):
    fut_df = con.execute(f"""
        SELECT "Timestamp", "Timestamp_Europe/Berlin", "MicroPrice_tick-based_10_1s" AS MicroPrice
        FROM read_csv('{parent}/data/raw/FUT_DAX Futures/FUT_DAX Futures_{date}.csv.gz')
    """).df()

    # filter needs the tz-aware datetime (Berlin wall-clock); resample_last then
    # collapses to one row per 100ms on the int-ns axis, cutting the feature-matrix
    # cost. Timestamps are not floored, so the later backward merge_asof is leak-free.
    fut_df = du.filter_trading_hours(fut_df)
    fut_df = du.resample_last(fut_df, "Timestamp", "100ms")

    fut = du.compute_feature_target_matrix(
        df=fut_df,
        ts_col='Timestamp',
        target_cols=[],
        feature_cols=['MicroPrice'],
        horizons=['-5m', '-2.5m', '-1m', '-30s', '-15s', '-10s', '-2s', '-1s', '-100ms'])

    fut_dict[date] = fut

Dates:   0%|          | 0/10 [00:00<?, ?it/s]

processed all fut dfs in: 80.54s


In [4]:
cols = ["Timestamp", "Timestamp_Europe/Berlin", "L1-BidPrice", "L1-AskPrice", "Side",
        "L1-QImb", "MidPrice", "MidPriceQW", "MidPriceCQW"]

symbols = [s for s in du.SYMBOLS[:1] if s != 'FUT_DAX_Futures']


for symbol in tqdm(symbols, desc="Symbols"):

    os.makedirs(f'{parent}/data/processed/{symbol}', exist_ok=True)


    for date in tqdm(du.SAMPLE_DATES[:10], desc="Dates", leave=False):
        path = f'{parent}/data/raw/CS_{symbol}/CS_{symbol}_{date}.csv.gz'

        group = con.execute(f"""
            SELECT {", ".join(f'"{c}"' for c in cols)}, "MicroPrice_tick-based_10_1s" AS MicroPrice
            FROM read_csv('{path}')
        """).df()

        filtered = du.filter_trading_hours(group).copy()

        filtered['TransactionPrice'] = du.compute_transaction_price(filtered)

        filtered = du.resample_last(filtered, "Timestamp", "100ms")

        featured = du.compute_feature_target_matrix(
            df=filtered,
            ts_col='Timestamp',
            target_cols=du.PRICE_MEASURES,
            feature_cols=['L1-QImb', 'MicroPrice'],
            horizons=['-5m', '-2.5m', '-1m', '-30s', '-15s', '-10s', '-2s', '-1s', '-100ms',
                        '100ms', '1s', '2s', '10s', '15s', '30s', '1m', '2.5m', '5m'])

        # merge_asof requires both sides sorted on the key (Timestamp, int ns).
        # Both come out of resample_last / compute_feature_target_matrix sorted,
        # so this is only a safety net.
        featured = featured.sort_values("Timestamp").reset_index(drop=True)

        merged = pd.merge_asof(
            featured,
            fut_dict[date].drop(columns="Timestamp_Europe/Berlin"),
            on="Timestamp",
            direction="backward",
            suffixes=("_LOB", "_FUT"))

        merged = merged.drop(columns=['TransactionPrice', 'MidPriceQW', 'MidPriceCQW',
                                        'MicroPrice_FUT', 'MicroPrice_LOB',
                                        'Timestamp_Europe/Berlin'])
        merged.to_parquet(f'{parent}/data/processed/{symbol}/{date}.parquet')

Symbols:   0%|          | 0/1 [00:00<?, ?it/s]

Dates:   0%|          | 0/10 [00:00<?, ?it/s]

In [5]:
test = pd.read_parquet(f"{parent}/data/processed/Adidas/2023-01-02.parquet")
test

,Timestamp,L1-BidPrice,L1-AskPrice,Side,L1-QImb,MidPrice,T_TransactionPrice_LogReturn_100ms,T_TransactionPrice_LogReturn_1s,T_TransactionPrice_LogReturn_2s,T_TransactionPrice_LogReturn_10s,...,F_MicroPrice_-100ms_LOB,F_MicroPrice_-5m_FUT,F_MicroPrice_-2.5m_FUT,F_MicroPrice_-1m_FUT,F_MicroPrice_-30s_FUT,F_MicroPrice_-15s_FUT,F_MicroPrice_-10s_FUT,F_MicroPrice_-2s_FUT,F_MicroPrice_-1s_FUT,F_MicroPrice_-100ms_FUT
0,1672647420064020680,127.22,127.34,NaN,0.553571,127.28,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1672647420348107143,127.22,127.34,NaN,0.553571,127.28,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1672647420414969868,127.22,127.34,NaN,0.553571,127.28,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1672647420815036370,127.22,127.34,NaN,0.553571,127.28,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000023
4,1672647421761958987,127.28,127.36,NaN,0.685484,127.32,0.0,0.0,0.0,0.000628,...,0.000323,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000019,0.000008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43114,1672676098020415330,127.66,127.72,NaN,0.615561,127.69,0.0,0.0,NaN,NaN,...,0.000000,0.000177,-0.000107,0.000224,-0.000146,-0.000003,-0.000021,-0.000013,-0.000024,-0.000033
43115,1672676099354530666,127.66,127.72,NaN,0.614155,127.69,0.0,NaN,NaN,NaN,...,0.000000,0.000177,-0.000149,0.000272,-0.000151,-0.000071,0.000010,-0.000033,0.000000,0.000005
43116,1672676099417232188,127.66,127.72,NaN,0.648871,127.69,0.0,NaN,NaN,NaN,...,0.000010,0.000177,-0.000149,0.000272,-0.000151,-0.000071,0.000010,-0.000033,0.000000,0.000005
43117,1672676099633935077,127.66,127.72,NaN,0.650206,127.69,0.0,NaN,NaN,NaN,...,0.000000,0.000177,-0.000149,0.000272,-0.000151,-0.000071,0.000010,-0.000033,0.000000,0.000005
